# 04B — Final Test Candidate Generation — FSP Locked Exact — FINAL V1

Notebook ini memakai **hasil lock dari `02C`**, bukan melakukan tuning ulang.

Konfigurasi dibaca langsung dari:
`Config/fsp_locked_config.json`

Untuk hasil penelitian saat ini, notebook mengharapkan method **Exact Fragment-Shot Prompting**. `max_fragment_len` dan `k_per_fragment` dibaca dari file lock (jadi tidak ditetapkan ulang berdasarkan test).

Protokol final:
- retrieval **hanya dari 417 data train**;
- longest-first exact fragment, non-overlap;
- train row dengan **full source identik** terhadap query test dikeluarkan dari retrieval;
- test reference Indonesia **tidak dibaca sama sekali**;
- 53 test × 5 kandidat;
- seed `42,43,44,45,46`;
- `temperature=0.70`, `top_p=0.95`, `top_k=0`, `num_beams=1`;
- `max_new_tokens=128`;
- belum menjalankan S3E/MBR atau metrik reference-based.

Output utama:
`Hasil/final_generation/fsp_locked/run_<id>/candidates_long.csv`.


## 1. Install dependency

In [1]:
# ============================================================
# INSTALL — jalankan sekali pada instance Vast.ai baru
# ============================================================
# PyTorch/CUDA bawaan image TIDAK diinstal ulang.
%pip -q install -U --upgrade-strategy only-if-needed \
    "transformers==4.48.1" \
    "accelerate==1.2.1" \
    "huggingface_hub==0.27.1" \
    "pandas==2.2.3" \
    "numpy==1.26.4" \
    "tqdm>=4.66,<5"

print('Dependency selesai.')
print('Jika package berubah dan kernel meminta restart, restart SATU KALI lalu mulai lagi dari Environment.')


Note: you may need to restart the kernel to use updated packages.
Dependency selesai.
Jika package berubah dan kernel meminta restart, restart SATU KALI lalu mulai lagi dari Environment.


## 2. Environment + test-input audit + GPU preflight

In [2]:
from __future__ import annotations

import os, re, json, time, random, hashlib, gc, subprocess
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

# ============================================================
# PROJECT LOCK
# ============================================================
PROJECT_ROOT = Path(
    os.getenv(
        'SKRIPSI_PROJECT_ROOT',
        '/workspace/10122233_CikitaMeydilvira_Skripsi',
    )
).expanduser().resolve()

DATASET_DIR = PROJECT_ROOT / 'Dataset'
TRAIN_PATH = DATASET_DIR / 'train.csv'
TEST_PATH = DATASET_DIR / 'test.csv'
CONFIG_DIR = PROJECT_ROOT / 'Config'
HASIL_DIR = PROJECT_ROOT / 'Hasil'

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
DEVICE_INDEX = 0
DTYPE = torch.bfloat16

EXPECTED_TRAIN = 417
EXPECTED_TRAIN_CONV = 44
EXPECTED_TEST = 53
EXPECTED_TEST_CONV = 3

# ============================================================
# FINAL MULTI-INSTANCE GENERATION LOCK
# ============================================================
CANDIDATE_SEEDS = [42, 43, 44, 45, 46]
TEMPERATURE = 0.70
TOP_P = 0.95
TOP_K = 0               # disable top-k; sampling dikontrol temperature + top_p
NUM_BEAMS = 1
MAX_NEW_TOKENS = 128
REPETITION_PENALTY = 1.0

os.environ.setdefault('HF_HOME', '/workspace/.hf_home')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

HF_TOKEN = get_token() or os.getenv('HF_TOKEN', '').strip() or None

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_ROOT tidak ditemukan: {PROJECT_ROOT}')

if not TEST_PATH.exists():
    raise FileNotFoundError(f'test.csv tidak ditemukan: {TEST_PATH}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA tidak tersedia.')

torch.cuda.set_device(DEVICE_INDEX)

if not torch.cuda.is_bf16_supported():
    raise RuntimeError('GPU tidak mendukung BF16.')

if not HF_TOKEN:
    raise RuntimeError('Hugging Face belum login. Jalankan `hf auth login` di terminal.')

# ------------------------------------------------------------
# GPU preflight: notebook final inference harus dimulai dari GPU bersih.
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)
free_gib = free_b / (1024 ** 3)
total_gib = total_b / (1024 ** 3)
print(f'GPU preflight: {free_gib:.2f} GiB free / {total_gib:.2f} GiB total')

if free_gib < 20.0:
    raise RuntimeError(
        f'VRAM bebas hanya {free_gib:.2f} GiB. '
        'Shutdown kernel notebook lain sebelum menjalankan final generation.'
    )

# ------------------------------------------------------------
# Test loader: target/reference SENGAJA tidak dibaca.
# ------------------------------------------------------------
TEST_INPUT_COLUMNS = [
    'row_id',
    'conversation_no',
    'turn_order',
    'duri_text',
]

test_df = pd.read_csv(
    TEST_PATH,
    usecols=TEST_INPUT_COLUMNS,
    dtype=str,
    keep_default_na=False,
    encoding='utf-8-sig',
)

test_df.columns = [str(c).strip() for c in test_df.columns]
for c in TEST_INPUT_COLUMNS:
    test_df[c] = test_df[c].astype(str).str.strip()

if len(test_df) != EXPECTED_TEST:
    raise ValueError(f'test.csv expected {EXPECTED_TEST}, found {len(test_df)}')

if test_df['row_id'].eq('').any() or test_df['row_id'].duplicated().any():
    raise ValueError('test.csv: row_id kosong/duplikat.')

if test_df['duri_text'].eq('').any():
    raise ValueError('test.csv: terdapat source Duri kosong.')

if test_df['conversation_no'].nunique() != EXPECTED_TEST_CONV:
    raise ValueError(
        f'Test conversation expected {EXPECTED_TEST_CONV}, '
        f'found {test_df["conversation_no"].nunique()}'
    )


def norm_text(x: str) -> str:
    return re.sub(r'\s+', ' ', str(x).strip().casefold())


def test_input_hash() -> str:
    h = hashlib.sha256()
    ordered = test_df.sort_values('row_id', kind='stable')
    for r in ordered.itertuples(index=False):
        payload = {
            'row_id': r.row_id,
            'conversation_no': r.conversation_no,
            'turn_order': r.turn_order,
            'duri_text': r.duri_text,
        }
        h.update(
            (json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(',', ':')) + '\n')
            .encode('utf-8')
        )
    return h.hexdigest()

TEST_INPUT_HASH = test_input_hash()


def terminator_ids(tokenizer) -> List[int]:
    ids = []
    if tokenizer.eos_token_id is not None:
        if isinstance(tokenizer.eos_token_id, (list, tuple)):
            ids.extend(int(x) for x in tokenizer.eos_token_id)
        else:
            ids.append(int(tokenizer.eos_token_id))

    for tok in ['<|eot_id|>', '<|end_of_text|>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if (
            isinstance(tid, int)
            and tid >= 0
            and tid != tokenizer.unk_token_id
            and tid not in ids
        ):
            ids.append(tid)
    return ids


def clean_translation(text: str) -> str:
    x = re.sub(r'\s+', ' ', str(text or '').strip())
    prefixes = [
        'Bahasa Indonesia:',
        'Indonesia:',
        'Terjemahan:',
        'Hasil terjemahan:',
        'Jawaban:',
    ]
    changed = True
    while changed:
        changed = False
        for p in prefixes:
            if x.casefold().startswith(p.casefold()):
                x = x[len(p):].strip()
                changed = True
                break
    return x


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

print('PROJECT_ROOT   :', PROJECT_ROOT)
print('TEST           :', len(test_df), 'rows /', test_df['conversation_no'].nunique(), 'conversation')
print('REFERENCE      : TIDAK DIBACA')
print('TEST INPUT HASH:', TEST_INPUT_HASH)
print('GPU            :', torch.cuda.get_device_name(DEVICE_INDEX))
print('MODEL          :', MODEL_ID)
print('SEEDS          :', CANDIDATE_SEEDS)
print('SAMPLING       : T=0.70, top_p=0.95, top_k=0, beams=1')


GPU preflight: 23.13 GiB free / 23.52 GiB total
PROJECT_ROOT   : /workspace/10122233_CikitaMeydilvira_Skripsi
TEST           : 53 rows / 3 conversation
REFERENCE      : TIDAK DIBACA
TEST INPUT HASH: 4d3e338a3effb23b17c0dca790808cd031072969484af3ffcb4f1bb9698242d2
GPU            : NVIDIA GeForce RTX 4090
MODEL          : meta-llama/Llama-3.1-8B-Instruct
SEEDS          : [42, 43, 44, 45, 46]
SAMPLING       : T=0.70, top_p=0.95, top_k=0, beams=1


## 3. Read locked FSP config + load train-only retrieval corpus

In [4]:
# ============================================================
# FSP LOCKED CONFIG + TRAIN CORPUS
# ============================================================
FSP_LOCK_PATH = CONFIG_DIR / 'fsp_locked_config.json'

if not FSP_LOCK_PATH.exists():
    raise FileNotFoundError(
        f'FSP locked config tidak ditemukan: {FSP_LOCK_PATH}. Jalankan 02C terlebih dahulu.'
    )

fsp_cfg = json.loads(FSP_LOCK_PATH.read_text(encoding='utf-8'))

if fsp_cfg.get('method') != 'fsp_exact_fragment':
    raise RuntimeError(
        'Notebook ini khusus FSP Exact. Config locked saat ini bukan fsp_exact_fragment: '
        f"{fsp_cfg.get('method')}"
    )

if fsp_cfg.get('model_id') != MODEL_ID:
    raise RuntimeError(
        f"Model locked FSP berbeda: {fsp_cfg.get('model_id')} != {MODEL_ID}"
    )

if bool(fsp_cfg.get('test_used_for_selection', False)):
    raise RuntimeError('Config FSP menyatakan test dipakai saat selection. Jangan lanjut.')

MAX_FRAGMENT_LEN = int(fsp_cfg['max_fragment_len'])
K_PER_FRAGMENT = int(fsp_cfg['k_per_fragment'])
RETRIEVAL_SEED = 42

TRAIN_COLUMNS = [
    'row_id',
    'conversation_no',
    'turn_order',
    'duri_text',
    'indonesia_text',
]

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'train.csv tidak ditemukan: {TRAIN_PATH}')

train_df = pd.read_csv(
    TRAIN_PATH,
    usecols=TRAIN_COLUMNS,
    dtype=str,
    keep_default_na=False,
    encoding='utf-8-sig',
)

for c in TRAIN_COLUMNS:
    train_df[c] = train_df[c].astype(str).str.strip()

if len(train_df) != EXPECTED_TRAIN:
    raise ValueError(f'train.csv expected {EXPECTED_TRAIN}, found {len(train_df)}')

if train_df['conversation_no'].nunique() != EXPECTED_TRAIN_CONV:
    raise ValueError(
        f'Train conversation expected {EXPECTED_TRAIN_CONV}, '
        f'found {train_df["conversation_no"].nunique()}'
    )

if train_df['row_id'].eq('').any() or train_df['row_id'].duplicated().any():
    raise ValueError('train.csv: row_id kosong/duplikat.')

if train_df['duri_text'].eq('').any() or train_df['indonesia_text'].eq('').any():
    raise ValueError('train.csv: terdapat teks kosong.')

# Conversation test harus disjoint dari train.
conv_overlap = sorted(set(train_df['conversation_no']) & set(test_df['conversation_no']))
if conv_overlap:
    raise RuntimeError(f'Conversation leakage train-test: {conv_overlap}')

print('FSP LOCKED METHOD    :', fsp_cfg['method'])
print('MAX FRAGMENT LEN     :', MAX_FRAGMENT_LEN)
print('K PER FRAGMENT       :', K_PER_FRAGMENT)
print('VALIDATION chrF++    :', fsp_cfg.get('validation_chrfpp'))
print('VALIDATION BLEU      :', fsp_cfg.get('validation_bleu'))
print('TRAIN RETRIEVAL ONLY :', len(train_df), 'rows')
print('TEST REFERENCE       : TIDAK DIBACA')


FSP LOCKED METHOD    : fsp_exact_fragment
MAX FRAGMENT LEN     : 7
K PER FRAGMENT       : 6
VALIDATION chrF++    : 29.3503
VALIDATION BLEU      : 6.6535
TRAIN RETRIEVAL ONLY : 417 rows
TEST REFERENCE       : TIDAK DIBACA


## 4. Build Exact Fragment index

In [ ]:
# ============================================================
# EXACT FRAGMENT RETRIEVAL
# ============================================================
MIN_FRAGMENT_LEN = 1


def toks(text: str) -> List[str]:
    return norm_text(text).split()

# inverted index exact n-gram -> train row indices
exact_index: Dict[Tuple[str, ...], List[int]] = {}

for idx, r in tqdm(train_df.iterrows(), total=len(train_df), desc='Build exact fragment index'):
    tt = toks(r['duri_text'])
    seen = set()
    for n in range(1, min(MAX_FRAGMENT_LEN, len(tt)) + 1):
        for s in range(len(tt) - n + 1):
            key = tuple(tt[s:s+n])
            if key in seen:
                continue
            seen.add(key)
            exact_index.setdefault(key, []).append(idx)


def stable_rng(*parts: Any) -> random.Random:
    raw = '||'.join(map(str, parts)).encode('utf-8')
    local_seed = int(hashlib.sha256(raw).hexdigest()[:16], 16) + RETRIEVAL_SEED
    return random.Random(local_seed)


def exact_retrieve(query_text: str) -> Dict[str, Any]:
    q = toks(query_text)
    query_norm = norm_text(query_text)
    occupied = set()
    fragments = []
    examples = []
    seen_rows = set()
    excluded_full_source_rows = []

    for n in range(min(MAX_FRAGMENT_LEN, len(q)), MIN_FRAGMENT_LEN - 1, -1):
        for start in range(0, len(q) - n + 1):
            pos = set(range(start, start+n))
            if pos & occupied:
                continue

            key = tuple(q[start:start+n])
            raw_matches = list(exact_index.get(key, []))

            matches = []
            for idx in raw_matches:
                if norm_text(train_df.loc[idx, 'duri_text']) == query_norm:
                    excluded_full_source_rows.append(str(train_df.loc[idx, 'row_id']))
                    continue
                matches.append(idx)

            if not matches:
                continue

            occupied |= pos
            frag = ' '.join(key)

            rng = stable_rng(query_text, frag, K_PER_FRAGMENT)
            rng.shuffle(matches)

            added = 0
            selected_ids = []
            for idx in matches:
                rid = str(train_df.loc[idx, 'row_id'])
                if rid in seen_rows:
                    continue

                examples.append({
                    'row_id': rid,
                    'fragment': frag,
                    'duri_text': str(train_df.loc[idx, 'duri_text']),
                    'indonesia_text': str(train_df.loc[idx, 'indonesia_text']),
                })
                seen_rows.add(rid)
                selected_ids.append(rid)
                added += 1

                if added >= K_PER_FRAGMENT:
                    break

            fragments.append({
                'fragment': frag,
                'start': start,
                'length': n,
                'available_train_rows_after_identical_source_filter': len(matches),
                'selected_example_count': added,
                'selected_row_ids': selected_ids,
            })

    return {
        'fragments': fragments,
        'examples': examples,
        'coverage': (len(occupied) / len(q)) if q else 0.0,
        'excluded_full_source_train_row_ids': sorted(set(excluded_full_source_rows)),
    }

print('Exact fragment index:', len(exact_index), 'unique fragments')

Build exact fragment index:   0%|          | 0/417 [00:00<?, ?it/s]

Exact fragment index: 9548 unique fragments


## 5. Load Llama

In [6]:
# ============================================================
# LOAD LLAMA 3.1 8B INSTRUCT — BF16, no quantization
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    device_map={'': DEVICE_INDEX},
)

model.eval()
model.config.use_cache = True
EOS_IDS = terminator_ids(tokenizer)

free_b, total_b = torch.cuda.mem_get_info(DEVICE_INDEX)
print('Llama loaded.')
print(f'Free VRAM: {free_b/(1024**3):.2f} GiB / {total_b/(1024**3):.2f} GiB')
print('Terminator IDs:', EOS_IDS)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama loaded.
Free VRAM: 8.17 GiB / 23.52 GiB
Terminator IDs: [128009, 128001]


## 6. FSP prompt + generation helper

In [ ]:
# ============================================================
# FSP PROMPT
# ============================================================
FSP_BASE_INSTRUCTION = (
    'Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia. '
    'Gunakan contoh pasangan terjemahan yang diberikan sebagai konteks. '
    'Keluarkan hanya terjemahan bahasa Indonesia tanpa penjelasan tambahan.'
)


def build_fsp_prompt(source: str, retrieval: Dict[str, Any]) -> str:
    blocks = [FSP_BASE_INSTRUCTION, '']

    if retrieval['examples']:
        blocks.append('Contoh:')
        for i, e in enumerate(retrieval['examples'], start=1):
            blocks.extend([
                f'[Contoh {i}]',
                f"Duri: {e['duri_text']}",
                f"Indonesia: {e['indonesia_text']}",
                '',
            ])

    blocks.extend([
        'Teks yang harus diterjemahkan:',
        f'Duri: {source}',
        'Indonesia:',
    ])
    return '\n'.join(blocks).strip()


def encode_prompt(prompt: str) -> Dict[str, torch.Tensor]:
    enc = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    )
    return {k: v.to(f'cuda:{DEVICE_INDEX}') for k, v in enc.items()}


def generate_one_candidate(
    enc: Dict[str, torch.Tensor],
    seed: int,
) -> Tuple[str, int, float]:
    seed_everything(seed)
    prompt_tokens = int(enc['input_ids'].shape[-1])

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            num_beams=NUM_BEAMS,
            repetition_penalty=REPETITION_PENALTY,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=EOS_IDS if EOS_IDS else None,
        )

    torch.cuda.synchronize()
    latency = time.perf_counter() - t0
    completion_tokens = int(out.shape[-1] - prompt_tokens)

    raw = tokenizer.decode(
        out[0, prompt_tokens:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return clean_translation(raw), completion_tokens, latency


## 7. Generate 53 × 5 candidates + save retrieval audit

In [8]:
# ============================================================
# FINAL FSP LOCKED — 53 test × 5 candidates
# Retrieval: TRAIN 417 only.
# Test reference: TIDAK DIBACA.
# ============================================================
run_payload = {
    'scenario': 'fsp_locked_exact',
    'model_id': MODEL_ID,
    'test_input_hash': TEST_INPUT_HASH,
    'reference_read': False,
    'locked_config_path': str(FSP_LOCK_PATH),
    'locked_method': fsp_cfg['method'],
    'max_fragment_len': MAX_FRAGMENT_LEN,
    'k_per_fragment': K_PER_FRAGMENT,
    'retrieval_seed': RETRIEVAL_SEED,
    'retrieval_corpus': 'train.csv only',
    'identical_full_source_train_rows_excluded': True,
    'candidate_seeds': CANDIDATE_SEEDS,
    'generation': {
        'do_sample': True,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'top_k': TOP_K,
        'num_beams': NUM_BEAMS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'repetition_penalty': REPETITION_PENALTY,
    },
    'prompt_version': fsp_cfg.get('prompt_template_version', 'fsp_duri_indonesia_shared_prompt_v4'),
}

RUN_ID = hashlib.sha256(
    json.dumps(run_payload, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()[:16]

RUN_DIR = HASIL_DIR / 'final_generation' / 'fsp_locked' / f'run_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_PATH = RUN_DIR / 'candidates_long.csv'
WIDE_PATH = RUN_DIR / 'candidates_wide.csv'
RETRIEVAL_PATH = RUN_DIR / 'retrieval_audit.csv'
MANIFEST_PATH = RUN_DIR / 'run_manifest.json'
SUMMARY_PATH = RUN_DIR / 'generation_summary.json'

MANIFEST_PATH.write_text(
    json.dumps({**run_payload, 'run_id': RUN_ID, 'created_at_utc': utc_now_iso()}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

if CANDIDATES_PATH.exists():
    records = pd.read_csv(
        CANDIDATES_PATH,
        dtype={'row_id': str, 'seed': int},
        keep_default_na=False,
        encoding='utf-8-sig',
    ).to_dict('records')
else:
    records = []

if RETRIEVAL_PATH.exists():
    retrieval_records = pd.read_csv(
        RETRIEVAL_PATH,
        dtype=str,
        keep_default_na=False,
        encoding='utf-8-sig',
    ).to_dict('records')
else:
    retrieval_records = []

done = {(str(r['row_id']), int(r['seed'])) for r in records if str(r.get('output', '')).strip()}
retrieval_done = {str(r['row_id']) for r in retrieval_records}

for row in tqdm(test_df.itertuples(index=False), total=len(test_df), desc='FSP locked test'):
    source = str(row.duri_text)
    retrieval = exact_retrieve(source)
    prompt = build_fsp_prompt(source, retrieval)
    enc = encode_prompt(prompt)
    prompt_tokens = int(enc['input_ids'].shape[-1])

    if str(row.row_id) not in retrieval_done:
        retrieval_records.append({
            'row_id': str(row.row_id),
            'conversation_no': str(row.conversation_no),
            'turn_order': str(row.turn_order),
            'source': source,
            'coverage': float(retrieval['coverage']),
            'fragment_count': len(retrieval['fragments']),
            'example_count': len(retrieval['examples']),
            'prompt_tokens': prompt_tokens,
            'excluded_full_source_train_rows_count': len(retrieval['excluded_full_source_train_row_ids']),
            'excluded_full_source_train_row_ids_json': json.dumps(
                retrieval['excluded_full_source_train_row_ids'], ensure_ascii=False
            ),
            'fragments_json': json.dumps(retrieval['fragments'], ensure_ascii=False),
            'examples_json': json.dumps(retrieval['examples'], ensure_ascii=False),
        })
        pd.DataFrame(retrieval_records).drop_duplicates(
            subset=['row_id'], keep='last'
        ).sort_values('row_id', kind='stable').to_csv(
            RETRIEVAL_PATH, index=False, encoding='utf-8-sig'
        )
        retrieval_done.add(str(row.row_id))

    for candidate_id, seed in enumerate(CANDIDATE_SEEDS, start=1):
        key = (str(row.row_id), int(seed))
        if key in done:
            continue

        error = ''
        output = ''
        completion_tokens = 0
        latency = np.nan

        try:
            output, completion_tokens, latency = generate_one_candidate(enc, seed)
            if not output:
                raise RuntimeError('Output kosong.')
        except Exception as e:
            error = f'{type(e).__name__}: {e}'

        records.append({
            'scenario': 'fsp_locked_exact',
            'row_id': str(row.row_id),
            'conversation_no': str(row.conversation_no),
            'turn_order': str(row.turn_order),
            'source': source,
            'candidate_id': candidate_id,
            'seed': seed,
            'temperature': TEMPERATURE,
            'top_p': TOP_P,
            'top_k': TOP_K,
            'max_new_tokens': MAX_NEW_TOKENS,
            'max_fragment_len': MAX_FRAGMENT_LEN,
            'k_per_fragment': K_PER_FRAGMENT,
            'coverage': float(retrieval['coverage']),
            'fragment_count': len(retrieval['fragments']),
            'example_count': len(retrieval['examples']),
            'prompt_tokens': prompt_tokens,
            'completion_tokens': completion_tokens,
            'latency_seconds': latency,
            'output': output,
            'error': error,
        })

        ckpt = pd.DataFrame(records).drop_duplicates(
            subset=['row_id', 'seed'], keep='last'
        ).sort_values(['row_id', 'candidate_id'], kind='stable')
        ckpt.to_csv(CANDIDATES_PATH, index=False, encoding='utf-8-sig')

        if output:
            done.add(key)

    del enc
    gc.collect()

cand = pd.read_csv(CANDIDATES_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
cand['candidate_id'] = pd.to_numeric(cand['candidate_id'], errors='raise').astype(int)
success = cand['output'].astype(str).str.strip().ne('')
expected_total = EXPECTED_TEST * len(CANDIDATE_SEEDS)

if len(cand) != expected_total or int(success.sum()) != expected_total:
    bad = cand.loc[~success, ['row_id', 'candidate_id', 'seed', 'error']]
    if not bad.empty:
        display(bad)
    raise RuntimeError(
        f'Final FSP belum lengkap: rows={len(cand)}, success={int(success.sum())}, expected={expected_total}. '
        f'Run ulang notebook; progress akan di-resume dari {CANDIDATES_PATH}.'
    )

retrieval_df = pd.read_csv(RETRIEVAL_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
if len(retrieval_df) != EXPECTED_TEST or retrieval_df['row_id'].nunique() != EXPECTED_TEST:
    raise RuntimeError('retrieval_audit belum lengkap 53/53.')

wide = cand.pivot(index='row_id', columns='candidate_id', values='output').reset_index()
wide = wide.rename(columns={i: f'candidate_{i}' for i in range(1, 6)})
wide = test_df[['row_id', 'conversation_no', 'turn_order', 'duri_text']].merge(wide, on='row_id', how='left')
wide.to_csv(WIDE_PATH, index=False, encoding='utf-8-sig')

summary = {
    'scenario': 'fsp_locked_exact',
    'run_id': RUN_ID,
    'locked_method': fsp_cfg['method'],
    'max_fragment_len': MAX_FRAGMENT_LEN,
    'k_per_fragment': K_PER_FRAGMENT,
    'test_rows': EXPECTED_TEST,
    'candidates_per_row': len(CANDIDATE_SEEDS),
    'candidate_rows': int(len(cand)),
    'retrieval_rows': int(len(retrieval_df)),
    'complete': True,
    'reference_read': False,
    'retrieval_corpus': 'train.csv only',
    'test_input_hash': TEST_INPUT_HASH,
    'candidates_path': str(CANDIDATES_PATH),
    'retrieval_audit_path': str(RETRIEVAL_PATH),
    'wide_path': str(WIDE_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nFSP LOCKED FINAL GENERATION: COMPLETE 53 × 5')


FSP locked test:   0%|          | 0/53 [00:00<?, ?it/s]

{
  "scenario": "fsp_locked_exact",
  "run_id": "14c344bdb72c6065",
  "locked_method": "fsp_exact_fragment",
  "max_fragment_len": 7,
  "k_per_fragment": 6,
  "test_rows": 53,
  "candidates_per_row": 5,
  "candidate_rows": 265,
  "retrieval_rows": 53,
  "complete": true,
  "reference_read": false,
  "retrieval_corpus": "train.csv only",
  "test_input_hash": "4d3e338a3effb23b17c0dca790808cd031072969484af3ffcb4f1bb9698242d2",
  "candidates_path": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked/run_14c344bdb72c6065/candidates_long.csv",
  "retrieval_audit_path": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked/run_14c344bdb72c6065/retrieval_audit.csv",
  "wide_path": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked/run_14c344bdb72c6065/candidates_wide.csv"
}

FSP LOCKED FINAL GENERATION: COMPLETE 53 × 5
